# Task 1: Understanding the MaxText Data Formats and Data Input Pipelines

**Maxtext Library Github Link:** [AI-Hypercomputer/maxtext](https://github.com/AI-Hypercomputer/maxtext)

Maxtext Uses 4 Data Input Formats :


*   Grain
*   Huggingface Datasets


*   TFDS Tensorflow Datasets
*   Synthetic Datasets

One of the common misconceptions of these formats is that these are file formats. Even I've had it for a long time, instead these formats Mainly Represent different dataset-loading engine libraries/ methods



1.   Grain - Grain is a dataset loading library made for JAX by Google. Grain adresses a bottleneck in JAX's data loading pipeline, it's that JAX's distributed compute is so fast that a slow or a non-deterministic data loading pipeline can become a really big time bottleneck, They solve it by their design, which allows Grain to resume training deterministically from exactly where it stopped. This Makes it Deterministic and resilient to preemption. [google/grain](https://github.com/google/grain)
Grain Allows the following file formats:
        

*    [ArrayRecord](https://github.com/google/array_record): These Support Random Access and With arrayrecord: fully deterministic, resilient to preemption; global shuffle
*   TFRecords and Parquets: Stark difference is unlike Array Record these only support sequential access and With TFRecords/parquet: performant; fully deterministic, resilient to preemption; hierarchical shuffle.

    The docs dont mention any limitation for Grain input data formats

2.   Huggingface Datasets: datasets in Hugging Face Hub local/Cloud Storage datasets in json, parquet, arrow, csv, txt (sequential access). These are mainly used for convinience as no download is needed and are non deterministic with preemption and have very limited scalability.

1.   TFDS: These Store data in TFRecords, which is sequentially accessible, if we need a data point at (x+1) we will need to have all data points until x accessed. these are also non deterministic with preemption.
We can see the info and guides about these in [maxtext/docs/guides/data_input_pipeline.md](https://github.com/AI-Hypercomputer/maxtext/blob/main/docs/guides/data_input_pipeline.md)
2.   Synthetic Dataset: the first thing you notice about synthetic datsets is that there is no mention of this in [data_input_pipeline.md](https://github.com/AI-Hypercomputer/maxtext/blob/main/docs/guides/data_input_pipeline.md), Instead we find a reference of this in [maxtext/src/maxtext/configs/base.yml](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/configs/base.yml), and the functions related to this in [maxtext/src/maxtext/input_pipeline/synthetic_data_processing.py](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/input_pipeline/synthetic_data_processing.py) and reading the class SyntheticDataIterator and the function in it raw_generate_synthetic_data, what it does is create a synthetic datasets to train LMs, synthetic data as we can see in the class, creates random tokens, as we can see in this line

```
tokens = jax.random.randint(
        jax.random.PRNGKey(0),
        (config.global_batch_size_to_load, config.max_target_length + 1),
        0,
        config.vocab_size,
        dtype=jnp.int32,
    )
```
later in the function JIT Function call we also see
Per-step, raw_generate_synthetic_data:




```
output["inputs"] = tokens[:, :-1]    # all tokens except the last
output["targets"] = tokens[:, 1:]    # all tokens except the first (shifted by 1)
```



This is the standard next-token-prediction framing every LM is trained on: the token at position x in inputs should predict the token at position x in targets, which is actually the original token at position x+1. Slicing off the last vs first element of the same sequence




so, since `self.data` never changes and the function never changes, jit traces and compiles once, then every one of our 50 steps just replays that cached compiled program with zero new random-number generation or I/O. That means our step-time measurements are purely about model compute on that backend, no disk read, no network download, no dataset-shuffling overhead that can potentially add extra time overhead that can hinder our comparison, i think this is the real reason why synthetic data has to be used throughout the task.






In [ ]:
!git clone https://github.com/AmithisCurious/zenteiq-maxtext-assignment
!git clone https://github.com/AI-Hypercomputer/maxtext
#!cd maxtext && pip install -e .[cuda12] #For GPU
!cd maxtext && pip install -e .[tpu] #For TPU
#!install_tpu_pre_train_extra_deps #For TPU
#!cd maxtext &&pip install -e . #For CPU
#!pip install uv #For cpu new, cuz the above one gave kept giving import error
#!cd maxtext && uv pip install -e .

In [ ]:
!cp zenteiq-maxtext-assignment/task2-qwen-dense/qwen3-1b.yml maxtext/src/maxtext/configs/models/qwen3-1b.yml
!ls maxtext/src/maxtext/configs/models/ | grep qwen

# Trial Training Run for MaxText

In [ ]:
!ls
!cd maxtext && python3 -m maxtext.trainers.pre_train.train \
  src/maxtext/configs/base.yml \
  base_output_directory=/content/zenteiq-maxtext-assignment/task2-qwen-dense/Qwen0.6B_50Steps_gpu \
  run_name=qwen06b-gpu-50steps \
  model_name=qwen3-0.6b \
  tokenizer_type=huggingface \
  tokenizer_path=Qwen/Qwen3-0.6B \
  dataset_type=synthetic \
  steps=50 \
  attention=dot_product \
  per_device_batch_size=2 \
  max_target_length=512 \
  dtype=bfloat16 \
  weight_dtype=bfloat16 \
  enable_checkpointing=false \
  async_checkpointing=false

## Task 2 - Training Qwen 0.6B Model on GPU/TPU/CPU, Scaling the model from 0.6->1B

Let us start with the Scaling of the model, as we can already see the yaml file of the 0.6B model already in the maxtext config folders.

Here is what we already know from Maxtext config files for Qwen3-0.6B:



```
base_emb_dim: 1024
base_num_query_heads: 16
base_num_kv_heads: 8
base_mlp_dim: 3072
base_num_decoder_layers: 28
head_dim: 128
mlp_activations: ["silu", "linear"] # "hidden_act": "silu" implies SwiGLU
vocab_size: 151936

decoder_block: "qwen3"

normalization_layer_epsilon: 1.0e-6
```

QWEN being from the same family as GPT Style models, contains decoder-only transformers. A decoder-only transformer is structured as: one embedding lookup table, followed by N identical repeating decoder blocks, followed by a final projection back to vocabulary.
So Suince every block has exact same internal shape/structure, i.e, emb_dim, mlp_dim, stays the same throughout the model and dont change layer to layer, i.e,  each block costs the same number of parameters regardless of depth. Therefore we can state,


> total_params = embedding_params + (num_layers × per_layer_params)

Each block has two main components that hold almost all the weights: the attention mechanism and the MLP(essentially a feed forward Neural Network). so based on this we can draw that,



> per_layer_params = attention_params + mlp_params

Each attention block contains 4 main projections Q,K,V,O aka Query, Key, Value, Output. All of these are different matrices. Before attention math (dot products, softmax) can happen, every token's 1024-dim embedding has to be projected into three roles: Query, Key, Value. Each of these is a learned matrix multiply. while the role of Output matrix is to project the result of attention back down to 1024-dim.

Initially I had thought that, just like any other attention mechanisms, Q, K, V would all be the same size.
but looking at attention_op.py and attention.py revealed that it uses grouped-query attention (GQA). instead of giving every query head its own key/value head, 16(base_num_query_heads) query heads share only 8(base_num_kv_heads) key/value heads. This is a memory/speed optimization fewer KV heads means a smaller KV cache at inference time and fewer parameters here.
Hence,


> Q: emb_dim × (num_query_heads × head_dim) = 1024 × (16×128) = 1024 × 2048 = 2,097,152


> K: emb_dim × (num_kv_heads × head_dim)    = 1024 × (8×128)  = 1024 × 1024 = 1,048,576


> V: emb_dim × (num_kv_heads × head_dim)    = 1024 × 1024     = 1,048,576


> O: (num_query_heads × head_dim) × emb_dim = 2048 × 1024     = 2,097,152


> attention_params = 2,097,152 + 1,048,576 + 1,048,576 + 2,097,152 = 6,291,456

We know, Qwen3 uses [SwiGLU](https://sebastianraschka.com/faq/docs/swiglu-modern-llms.html) Swiglu has 3 main components -


*   The Feature (Up) Projection: Extracts the raw content and candidate features from the input.
*   The Gate Projection: Determines what is important, often using the SiLU (Swish) activation function. The gate's output is multiplied by the feature projection. This overlays the gate onto the features, zeroing out irrelevant noise and passing only highly relevant information.

*   The down projection is the final linear layer in the Feed-Forward Network (FFN) block. It takes the dynamically gated features and projects them back down to the model's original hidden dimension size.


so calculating params in these layers we get:


> gate: emb_dim × mlp_dim = 1024 × 3072 = 3,145,728


> up:   emb_dim × mlp_dim = 1024 × 3072 = 3,145,728


> down: mlp_dim × emb_dim = 3072 × 1024 = 3,145,728





> mlp_params = 3 × 3,145,728 = 9,437,184

Now that we have MLP Parameters and Attention parameters we can find per_layer_params


> per_layer_params = attention_params + mlp_params = 6,291,456 + 9,437,184 = 15,728,640





> embedding_params = vocab_size × emb_dim = 151,936 × 1024 = 155,582,464



Now plugging all of this into the initial formula we get:



> total_params(N) = embedding_params + N × per_layer_params
                = 155,582,464 + N × 15,728,640

For the base model, N = 28:




> total_params(28) = 155,582,464 + 28 × 15,728,640 = 155,582,464 + 440,459,264
                  = 596,041,728  ≈ 0.596 billion

This confirms the working of the formula as base model does have 0.6B parameters.
So in order to find fot 1B params we just have to solve for N.


> 1,000,000,000 ≤ 155,582,464 + N × 15,728,640


> N ≥ (1,000,000,000 − 155,582,464) / 15,728,640
N ≥ 53.65 or **N = 54**

To validate the updated model size,

> total_params(54) = 155,582,464 + 54 × 15,728,640 = 1,005,039,616  ≈ 1.005 billion





Below is the script that automates this training(Although I do have to change instance types manually). The script automates documentation and logging for each run on each instance type, i.e, 0.6b and 1b on training on all 3 instances, i.e, 6 trainings.

So all it takes to scale up this model is


```
"base_num_decoder_layers=54"
```

this is what we came to conclusion with in the above calculation the updated yaml file can be found in task2-qwen-dense/qwen3-1b.yaml


In [ ]:
import json
import subprocess, re, os, csv
import matplotlib.pyplot as plt

REPO_DIR = "/content/zenteiq-maxtext-assignment"
MAXTEXT_DIR = "/content/maxtext"

INSTANCE_TYPE = "gpu"  # "cpu", "gpu", or "tpu"

MODELS = {
    "0.6b": ("qwen3-0.6b", 4),
    "1b":   ("qwen3-0.6b", 4),
}

STEP_RE = re.compile(
    r"completed step: (\d+), seconds: ([\d.]+), TFLOP/s/device: ([\d.]+), "
    r"Tokens/s/device: ([\d.]+), total_weights: (\d+), loss: ([\d.]+), "
    r"lm_loss: ([\d.]+), perplexity: ([\d.]+|inf)"
)

SUMMARY_PATH = os.path.join(REPO_DIR, "task2-qwen-dense", "results_summary.json")

def record_summary(model_label, instance_type, model_name, rows):
    steady = [r for r in rows if r["step"] >= 5]
    if not steady:
        steady = rows

    entry = {
        "model_label": model_label,
        "instance_type": instance_type,
        "model_name": model_name,
        "num_steps_logged": len(rows),
        "avg_seconds_per_step": sum(r["seconds"] for r in steady) / len(steady),
        "avg_tflops_per_device": sum(r["tflops_per_device"] for r in steady) / len(steady),
        "avg_tokens_per_sec_device": sum(r["tokens_per_sec_device"] for r in steady) / len(steady),
        "final_loss": rows[-1]["loss"],
        "final_perplexity": rows[-1]["perplexity"],
    }

    existing = []
    if os.path.exists(SUMMARY_PATH):
        with open(SUMMARY_PATH) as f:
            existing = json.load(f)

    existing = [e for e in existing if not (e["model_label"] == model_label and e["instance_type"] == instance_type)]
    existing.append(entry)

    with open(SUMMARY_PATH, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"Recorded summary -> {SUMMARY_PATH}")

def run_training(model_label, model_name, batch_size):
    out_dir = os.path.join(REPO_DIR, "task2-qwen-dense", INSTANCE_TYPE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    run_name = f"{model_label}-{INSTANCE_TYPE}-50steps"

    cmd = [
        "python3", "-m", "maxtext.trainers.pre_train.train",
        "src/maxtext/configs/base.yml",
        f"base_output_directory={out_dir}/output",
        f"run_name={run_name}",
        f"model_name={model_name}",
        "tokenizer_type=huggingface",
        "tokenizer_path=Qwen/Qwen3-0.6B",
        "dataset_type=synthetic",
        "steps=50",
        f"per_device_batch_size={batch_size}",
        "max_target_length=512",
        "dtype=bfloat16",
        "weight_dtype=bfloat16",
        "attention=dot_product",
        f"hardware={INSTANCE_TYPE}",
        "enable_checkpointing=false",
        "async_checkpointing=false",

    ]
    if model_label == "1b":
      cmd.extend(["override_model_config=true","base_num_decoder_layers=54"])
    if INSTANCE_TYPE == "cpu":
      cmd.append("skip_jax_distributed_system=True")
    print(f"\n=== Running {model_label} on {INSTANCE_TYPE} (batch={batch_size}) ===")
    result = subprocess.run(cmd, cwd=MAXTEXT_DIR, capture_output=True, text=True)
    full_log = result.stdout + "\n" + result.stderr

    log_path = os.path.join(out_dir, "training.log")
    with open(log_path, "w") as f:
        f.write(full_log)
    print(f"Saved log -> {log_path}")

    rows = []
    for m in STEP_RE.finditer(full_log):
        step, seconds, tflops, tokens_s, total_w, loss, lm_loss, ppl = m.groups()
        rows.append({
            "step": int(step), "seconds": float(seconds),
            "tflops_per_device": float(tflops),
            "tokens_per_sec_device": float(tokens_s),
            "loss": float(loss), "lm_loss": float(lm_loss),
            "perplexity": float(ppl) if ppl != "inf" else float("inf"),
        })

    if not rows:
        print("WARNING: no step lines parsed -- open training.log, this run likely errored out.")
        return None

    csv_path = os.path.join(out_dir, "metrics.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved metrics -> {csv_path}")

    steps = [r["step"] for r in rows]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(steps, [r["loss"] for r in rows]); axes[0].set_title("loss"); axes[0].set_xlabel("step")
    axes[1].plot(steps, [r["seconds"] for r in rows]); axes[1].set_title("seconds/step"); axes[1].set_xlabel("step")
    axes[2].plot(steps, [r["tokens_per_sec_device"] for r in rows]); axes[2].set_title("tokens/s/device"); axes[2].set_xlabel("step")
    fig.suptitle(f"{model_label} on {INSTANCE_TYPE}")
    fig.tight_layout()
    plot_path = os.path.join(out_dir, "metrics.png")
    fig.savefig(plot_path)
    plt.show()
    print(f"Saved plot -> {plot_path}")
    record_summary(model_label, INSTANCE_TYPE, model_name, rows)
    return rows


for label, (name, bs) in MODELS.items():
    run_training(label, name, bs)

Script below creates the table and saves it under task2-qwen-dense in the github repo

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open(os.path.join(REPO_DIR, "task2-qwen-dense", "results_summary.json")) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df = df.sort_values(["model_label", "instance_type"])
df = df[["model_label", "instance_type", "avg_seconds_per_step", "avg_tflops_per_device", "avg_tokens_per_sec_device", "final_loss"]]
df.columns = ["Model", "Backend", "Avg sec/step", "Avg TFLOP/s/device", "Avg Tokens/s/device", "Final loss"]

fig, ax = plt.subplots(figsize=(10, len(df) * 0.6 + 1))
ax.axis("off")
table = ax.table(cellText=df.round(3).values, colLabels=df.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
fig.tight_layout()
fig.savefig(os.path.join(REPO_DIR, "task2-qwen-dense", "comparison_table.png"), dpi=150, bbox_inches="tight")
plt.show()

Maxtext mainly uses JAX which uses XLA for computations. which uses JIT functions to compile each train step to the bare bone hardware so The same Python/JAX code runs on all three backends because XLA handles the backend-specific details and adjustments.
The performance difference isn't because MaxText has three separate codepaths, it's because XLA maps the same matmul operations to radically different hardware execution models.  
TPU: Purpose-built for dense matmul. This is exactly the operation profile of a transformer. Hence ~90 TFLOP/s.
GPU (Tesla T4): General-purpose parallel processor. Good at matmuls via CUDA cores and Tensor Cores, but not architecturally specialized. Hence ~2.76 TFLOP/s, roughly 33× slower than TPU.
CPU: No matrix-multiply acceleration at all. and is very slow(seriously the CPU Training for these 50 steps took 2 hours and 15 mins on this colab notebook) Hence ~0.035 TFLOP/s

but all of this shouldnt affect the loss as a whole right? yes it is true. but we still find slightly varying loss values for 2 different runs on the same model on 2 different acceleration devices(GPU vs TPU), like if you see the table 0.6b model on gpu had a loss of 76.83 while on tpu it has a loss of 79.12, which ideally should not happen, because it is the same model, it has the exact same arithematic and matrix operations happen, although I cannot find any concrete reason to why this is happening, I assume it is because of floating point errors, as both losses are very close by even in multiple set of runs.

Reference - https://medium.com/@neurogenou/gpu-vs-tpu-understanding-the-differences-in-ai-training-and-inference-2e61e418c3a7

We notice differences between the 2 models due to the difference in number of layers (28-54) so, More layers = more sequential matmuls per forward+backward pass = more wall-clock time per step. Expected ratio: 54/28 ≈ 1.93×

Now we also see that the loss changes between 0.6b to 1b models, infact it reduces significantly actually,
This might seem odd for a synthetic data run, but it makes sense — a larger model has more parameters and therefore more capacity to memorize even a random synthetic sequence in the same number of steps. It's fitting noise faster.

There is also another thing worth noting since we are using a synthetic dataset,
Training on a synthetically generated random token sequence from randomly initialized weights means neither the loss values nor the perplexity numbers reflect model quality, because we are literally tying to converge onto random noise in this case

on final notes I'd like to menion one of my failure cases as well with colab free tier

Something worth noting for the CPU runs. while the 0.6B model ran fine on CPU (just very slowly, frustratingly slowly), the 1B model simply crashed, not with a Python error but with a silent OOM kill from the OS. The instinct here was to reduce per_device_batch_size and max_target_length to free up memory, but that didn't help no matter how low we went, and the reason is pretty straightforward once you look at XLA's memory breakdown,, Argument size: 5.6GB + Output size: 5.6GB = 11.2GB, which is the fixed cost of model parameters + gradients + Adam's two momentum buffers, and this number doesn't move regardless of batch size or sequence length since it only scales with parameter count. With free tier Colab giving ~12-13GB of host RAM, there's barely 1GB left over for anything else, and the process just gets killed. So this isn't a tuning problem, at 1B parameters, the optimizer state alone makes CPU training infeasible on standard consumer hardware, and the only real fixes would be a high-RAM machine, a memory-efficient optimizer like SGD (which eliminates the momentum buffers), or quantized training. we can see a clear breakdown of this in training logs of 1b model on cpu.

In [ ]:
from google.colab import userdata
import subprocess

GH_TOKEN = userdata.get('GH_TOKEN')
REPO_DIR = "/content/zenteiq-maxtext-assignment"
REMOTE_URL = f"https://{GH_TOKEN}@github.com/AmithisCurious/zenteiq-maxtext-assignment.git"
def push_results(instance_type):
    def run(cmd):
        result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        print(f"$ {' '.join(cmd if 'GH_TOKEN' not in str(cmd) else ['git','remote','set-url','origin','<hidden>'])}")
        print(result.stdout)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
        return result.returncode

    run(["git", "config", "user.email", "amithkumar8536@gmail.com"])
    run(["git", "config", "user.name", "AmithisCurious"])
    run(["git", "remote", "set-url", "origin", REMOTE_URL])
    run(["git", "add", "-A"])
    run(["git", "commit", "-m", f"add {instance_type} training logs for task 2"])
    rc = run(["git", "push", "origin", "main"])

    print("push succeeded" if rc == 0 else "push FAILED, see STDERR above")

push_results(INSTANCE_TYPE)

# Task 3: Working with Deepseek and MOE(Mixture of Experts) exploration

From the deepseek config file in maxtext we have:


```
base_emb_dim: 2048
base_num_query_heads: 16
base_num_kv_heads: 16
base_mlp_dim: 10944
base_moe_mlp_dim: 1408
base_num_decoder_layers: 27
first_num_dense_layers: 1
mlp_activations: ["silu","linear"]
vocab_size: 102400
enable_dropout: false
logits_via_embedding: false
normalization_layer_epsilon: 1.0e-6
num_experts: 64
num_experts_per_tok: 6
shared_experts: 2
routed_scaling_factor: 1.0
routed_score_func: "softmax"
routed_bias: false
decoder_block: "deepseek"
```

since `logits_via_embedding: false` there are 2 different matrices/ projections being used for hence for embedding params we will have to count it twice as they are identical projections.


> embedding_params = vocab_size × emb_dim x 2 = 102,400 × 2048 × 2  = 419,430,400



> dense MLP (1 layer): 3(no of projections in SwiGLU) x emb_dim x mlp_dim = 3 × 2048 × 10,944   =  67,239,936

to find the number of params in the MOE Layers we find the number of params per expert's MLP and multiply that with (num_experts + shared_experts) and multiply that with number of MOE Layers which is (base_num_decoder_layers - first_num_dense_layers)


> per expert MLP: 3(no of projections in SwiGLU) x emb_dim x moe_mlp_dim = 3 × 2048 × 1,408    =   8,650,752



> (num_experts + shared_experts) = 64 + 2 = 66



> (base_num_decoder_layers - first_num_dense_layers) = 27 - 1 = 26


hence,


> Total MOE Layer Params = 8,650,752 x 66 x 26 = 14,844,690,432

Please note I havent mentioned anything about attention parameters yet, Actually Deepseek uses MLA Attention method, the math behind which is something I am a little shaky about and currently do not have the time to go deeper into, if I have the chance before deadline will come back here and solve it for attention params as well but for now we will just call it `attn`



> Total params (excl. attention) = 419,430,400 + 67,239,936 + 14,844,690,432
                        ≈ 15331360768 parameters



> Total with attn will come up to = 15331360768 + 27*attn

where attn is total no of attention params, but for now the math somewhat checks out

in order to reduce this model to 1B  we will need to reduce the values uniformly, reducing just One param like num_layers or num_experts wont do the trick as model will still be >1B params

we will have to reduce multiple values simultaneously to make this happen.
Here are the following knobs we can play around with:
emb_dim, num_experts, base_moe_mlp_dim, num_layers

reducing too much of each one of these impact the model negatively in one way or the other.

so inorder to do this efficiently let us reduce num_layers first as there are 27 layers, let's make an arbitrary reduction of num_layers to 16 and reduce emb_dim to 1024, reducing emb_dim helps reduce a lot of params because it is used everywhere in all of model's calculations.

With num_layers=16 (15 MoE layers), emb_dim=1024, shared_experts=2:



> per expert MLP cost:  3 × 1024 × 1408 = 4,325,376



> embedding (×2):       102,400 × 1024 × 2 = 209,715,200




> dense MLP (1 layer):  3 × 1024 × 10,944  =  33,619,968

> remaining budget:     1,000,000,000 − 209,715,200 − 33,619,968 = 756,664,832





> 15 × (N + 2) × 4,325,376 ≤ 756,664,832

> (N + 2) ≤ 756,664,832 / 64,880,640 == (N + 2) ≤ 11.66

> N ≤ 9.66 or N = 8 (round to nearest power of 2)

The reason why I rounded off to 8 is because if we round it off to 9 we get model params as 967M, and this is excluding attention params, and if we do add it, then there is a good chance we will go over the 1B param mark.

So New total = 15 × 43,253,760 + 3 × 1024 × 10944 + 102400 × 1024 × 2 = 892,141,568



















In [ ]:
#!pip uninstall -y transformer-engine
#!pip uninstall -y torch torchvision torchaudio
!pip install pathwaysutils

In [ ]:
import json
import subprocess, re, os, csv
import matplotlib.pyplot as plt
from google.colab import userdata

REPO_DIR    = "/content/zenteiq-maxtext-assignment"
MAXTEXT_DIR = "/content/maxtext"
TASK_DIR    = os.path.join(REPO_DIR, "task3-deepseek-moe")

INSTANCE_TYPE = "cpu"  # change to "cpu", "gpu", or "tpu"

STEP_RE = re.compile(
    r"completed step: (\d+), seconds: ([\d.]+), TFLOP/s/device: ([\d.]+), "
    r"Tokens/s/device: ([\d.]+), total_weights: (\d+), loss: ([\d.]+), "
    r"lm_loss: ([\d.]+), perplexity: ([\d.]+|inf)"
)

SUMMARY_PATH = os.path.join(TASK_DIR, "results_summary.json")

def record_summary(instance_type, rows):
    steady = [r for r in rows if r["step"] >= 5] or rows
    entry = {
        "model": "deepseek-sub1b",
        "instance_type": instance_type,
        "num_steps_logged": len(rows),
        "avg_seconds_per_step":       sum(r["seconds"] for r in steady) / len(steady),
        "avg_tflops_per_device":      sum(r["tflops_per_device"] for r in steady) / len(steady),
        "avg_tokens_per_sec_device":  sum(r["tokens_per_sec_device"] for r in steady) / len(steady),
        "final_loss":       rows[-1]["loss"],
        "final_perplexity": rows[-1]["perplexity"],
    }
    existing = []
    if os.path.exists(SUMMARY_PATH):
        with open(SUMMARY_PATH) as f:
            existing = json.load(f)
    existing = [e for e in existing if e["instance_type"] != instance_type]
    existing.append(entry)
    with open(SUMMARY_PATH, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"Summary saved -> {SUMMARY_PATH}")

def run_training():
    out_dir  = os.path.join(TASK_DIR, INSTANCE_TYPE)
    run_name = f"deepseek-sub1b-{INSTANCE_TYPE}-50steps"
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        "python3", "-m", "maxtext.trainers.pre_train.train",
        "src/maxtext/configs/base.yml",
        f"base_output_directory={out_dir}/output",
        f"run_name={run_name}",
        "model_name=deepseek2-16b",
        "tokenizer_type=huggingface",
        "tokenizer_path=deepseek-ai/DeepSeek-V2-Lite",
        "dataset_type=synthetic",
        "steps=50",
        "per_device_batch_size=1",
        "max_target_length=512",
        "dtype=bfloat16",
        "weight_dtype=bfloat16",
        "attention=dot_product",
        f"hardware={INSTANCE_TYPE}",
        "enable_checkpointing=false",
        "async_checkpointing=false",
        "megablox=false",
        # scaling overrides — brings model to ~892M params
        "override_model_config=true",
        "base_emb_dim=1024",
        "base_num_decoder_layers=16",
        "num_experts=8",
        "num_experts_per_tok=2",
    ]

    if INSTANCE_TYPE == "cpu":
        cmd.extend(["skip_jax_distributed_system=True", "megablox=false"])

    print(f"\n=== Running deepseek-sub1b on {INSTANCE_TYPE} ===")
    result = subprocess.run(cmd, cwd=MAXTEXT_DIR, capture_output=True, text=True)
    full_log = result.stdout + "\n" + result.stderr

    log_path = os.path.join(out_dir, "training.log")
    with open(log_path, "w") as f:
        f.write(full_log)
    print(f"Log saved -> {log_path}")

    rows = []
    for m in STEP_RE.finditer(full_log):
        perp = float("inf") if m.group(8) == "inf" else float(m.group(8))
        rows.append({
            "step":                 int(m.group(1)),
            "seconds":              float(m.group(2)),
            "tflops_per_device":    float(m.group(3)),
            "tokens_per_sec_device":float(m.group(4)),
            "loss":                 float(m.group(6)),
            "perplexity":           perp,
        })

    if not rows:
        print("ERROR: No completed steps in log. Check training.log.")
        return

    csv_path = os.path.join(out_dir, "metrics.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["step","seconds","tflops_per_device","tokens_per_sec_device","loss","perplexity"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"CSV saved -> {csv_path}")

    steps = [r["step"] for r in rows]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(steps, [r["loss"] for r in rows]);               axes[0].set_title("Loss");            axes[0].set_xlabel("Step")
    axes[1].plot(steps, [r["tflops_per_device"] for r in rows]);  axes[1].set_title("TFLOP/s/device");  axes[1].set_xlabel("Step")
    axes[2].plot(steps, [r["tokens_per_sec_device"] for r in rows]); axes[2].set_title("Tokens/s/device"); axes[2].set_xlabel("Step")
    plt.suptitle(f"deepseek-sub1b on {INSTANCE_TYPE}")
    plt.tight_layout()
    png_path = os.path.join(out_dir, "metrics.png")
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()

    record_summary(INSTANCE_TYPE, rows)

    # push to git
    GH_TOKEN  = userdata.get("GH_TOKEN")
    REMOTE    = f"https://{GH_TOKEN}@github.com/AmithisCurious/zenteiq-maxtext-assignment.git"
    def run_git(cmd_list, redact=False):
        r = subprocess.run(cmd_list, cwd=REPO_DIR, capture_output=True, text=True)
        display = ["git", "remote", "set-url", "origin", "<hidden>"] if redact else cmd_list
        print(f"$ {' '.join(display)}")
        print(r.stdout)
        if r.returncode != 0: print("STDERR:", r.stderr)
        return r.returncode

    run_git(["git", "config", "user.email", "amithkumar8536@gmail.com"])
    run_git(["git", "config", "user.name", "AmithKumarG5"])
    run_git(["git", "remote", "set-url", "origin", REMOTE], redact=True)
    run_git(["git", "add", "-A"])
    run_git(["git", "commit", "-m", f"add task3 deepseek-sub1b {INSTANCE_TYPE} training logs"])
    rc = run_git(["git", "push", "origin", "main"])
    print("push succeeded" if rc == 0 else "push FAILED")

run_training()


In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

REPO_DIR = "/content/zenteiq-maxtext-assignment"
TASK_DIR  = os.path.join(REPO_DIR, "task3-deepseek-moe")

with open(os.path.join(TASK_DIR, "results_summary.json")) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df = df.sort_values("instance_type")
df = df[["model","instance_type","avg_seconds_per_step","avg_tflops_per_device","avg_tokens_per_sec_device","final_loss","final_perplexity"]]
df.columns = ["Model","Backend","Avg sec/step","Avg TFLOP/s/device","Avg Tokens/s/device","Final Loss","Final Perplexity"]

print(df.to_markdown(index=False))

fig, ax = plt.subplots(figsize=(10, len(df)*0.6 + 1))
ax.axis("off")
table = ax.table(cellText=df.round(3).values, colLabels=df.columns, cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
fig.tight_layout()
fig.savefig(os.path.join(TASK_DIR, "comparison_table.png"), dpi=150, bbox_inches="tight")
plt.show()

The comparison table above is also saved at task3-deepseek-moe/comparison_table.png in the repo.

To bring the model under 1B, I changed exactly 4 things from the deepseek2-16b config. First, base_emb_dim from 2048 to 1024, this is the single biggest lever since it touches every weight matrix in the entire model. Second, base_num_decoder_layers from 27 to 16, because fewer layers means fewer MoE blocks stacked up. Third, num_experts from 64 to 8, which directly cuts the expert pool size. Fourth, num_experts_per_tok from 6 to 2, which controls how many experts are active per token during the forward pass.

On GPU, the model ran all 50 steps and final loss came down to 1.786 with a perplexity of 5.97. On TPU it also completed all 50 steps with final loss of 5.679 and perplexity of 292.8. CPU failed before a single step completed — the XLA compiler estimated it needed 17.5 GB (5.7 GB params + optimizer state, 5.7 GB activations, 6.1 GB temp buffers), which is more than what Colab's CPU runtime actually has available.

Comparing to the Qwen3 runs from Task 2, there are two things that stand out. TFLOP/s is lower for DeepSeek (1.15 on GPU vs 2.76 for Qwen3-0.6B) — this isn't the model being slower, it's MoE doing less compute per token by design, since only 2 out of 8 experts run for each token. The bigger surprise is how much better the loss is: Qwen3 never got below ~50 loss in 50 steps while DeepSeek got to 1.79.

We see that the numbers from Qwen  experiments and Deepseek Experiments differ in Avg TFLOP/s/device, infact Deepseek achieves just 1.149 TFLOP/s/device as compared to 2.76 that qwen could achieve, this is not model being slower it is MOE coming into action, In a MoE layer with 8 experts and 2 active, only 25% of the expert weights are used per forward pass. Dense transformers are arithmetic-intense and pack well onto tensor cores; MoE models have sparse activation patterns.





And we again face a Failure for CPU, with an OOM error.

# Final Summary

throughout the series of these experiments we see that tpu is the clear winner on both TFLOPs and tokens and CPU is not a viable training backend(CLEARLY)

In every run, Step 0 is dramatically different from subsequent steps. This is because JAX uses JIT (just-in-time) compilation: the first call to the train step triggers XLA to compile the computation graph into device-specific kernel code.

With identical hyperparameters, random seed, and synthetic data, you would expect identical loss trajectories. They are not. For Qwen3-0.6B, final loss is 76.83 on GPU, 79.12 on TPU, and 54.57 on CPU. To be honest, even I am perplexed as to why loss on CPU is that far off from GPU or TPU, but these discrepancies between GPU and TPU losses can be accredited to Floating Point Non-determinism.

Scaling Qwen3 from 0.6B to 1B was done by increasing `base_num_decoder_layers` from 28 to 54. In doing so Step time nearly doubled, as expected.
The 1B model's final loss (50.41 on GPU, 54.03 on TPU) is lower than the 0.6B model (76.83 on GPU, 79.12 on TPU). The lower loss reflects the larger model having more parameters to adapt to gradient updates.

The most striking difference between dense and MoE is training stability on synthetic data:

- Qwen3 models: loss starts at 235+ and never falls below 50 in 50 steps. Perplexity remains very high.
- DeepSeek MoE: loss drops from 12.04 to 1.79 in 50 steps. Perplexity falls from 170,295 to 5.97.

# Unexpected Findings(There are quite a few)

CPU training was expected to be slow, but 108 seconds per step and was more extreme than anticipated. The effective tokens/s of 9.45 means a single epoch over a modest dataset would take months on CPU. However, unlike GPU and TPU, CPU training reached a lower final loss.

Both ~1B models failed on CPU with XLA memory estimates in the 11.8–12.0 GB range. Despite very different architectures (dense vs MoE)

GPU and TPU produced different final loss values for DeepSeek MoE, A 3.18× difference in final loss between two hardware backends running the same model and data

and in final words I'd like to say that our assumption of keeping N = 8 was infact correct, the approximate we got for params for deepseek which was around 892 million was infact the correct assumption as MaxText logged `number parameters: 1.019B` for the "sub-1B" DeepSeek config.